# 2. RunnablePassthrough

`RunnablePassthrough` passes its input through **unchanged**. That sounds trivial — but it's the key to
keeping the *original* data alongside new results inside a chain, and its `.assign()` method is one of the
most-used tools in real RAG and multi-step pipelines.

---

## 1. Simple Definition

> **Kid version:** Imagine a conveyor belt with side-stations 🏭. `RunnablePassthrough` is a plain belt
> segment that **carries your item forward without touching it**. Its helper `.assign()` is a station
> that **stamps an extra sticker on the item while it keeps moving** — nothing is removed, something is
> added.

**Professional definition:** `RunnablePassthrough` is a Runnable that returns its input as-is. Used
inside a `RunnableParallel` (a dict of Runnables), it preserves the original input under a key while
sibling branches compute new values. `RunnablePassthrough.assign(**kwargs)` returns the input **merged
with** new keys computed by other Runnables.

```python
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke({"q": "hi"})                       # {"q": "hi"}  (unchanged)
RunnablePassthrough.assign(n=lambda x: len(x["q"])).invoke({"q": "hi"})
# {"q": "hi", "n": 2}   ← original kept, new key added
```

---

## 2. Why Does It Exist?

**The problem:** In a **parallel** step (a dict of Runnables), each branch's output *replaces* the input
with a new dict. But downstream you often still need the **original** input (e.g., the user's question)
*and* the new computed values (e.g., retrieved context). Without a "keep this" tool, the original is lost.

### Before (the original input gets lost)

```python
# We retrieve context, but now we've thrown away the original question!
setup = {"context": retriever}          # → {"context": [...docs...]}   ...where's the question?
```

### After (passthrough keeps the original)

```python
from langchain_core.runnables import RunnablePassthrough

setup = {
    "context":  retriever,               # question → docs
    "question": RunnablePassthrough(),   # question → itself (kept!)
}
# → {"context": [...docs...], "question": "the original question"}
```

`RunnablePassthrough` exists to **carry data forward** through parallel/multi-step chains so later steps
can see both the input and the newly computed fields.

**Where you'll use it:** the classic **RAG** input shape, classify-then-branch routing, and any
multi-stage chain where a later step needs an earlier value.

---

## 3. Real-Life Analogy

**Photocopying a form while filling in new boxes** 📄. `RunnablePassthrough` hands the form onward
exactly as it arrived. `.assign()` fills in an extra box (say, "date received") **without erasing**
anything the customer already wrote — then passes the fuller form along.

---

## 4. Where It Fits in LangChain Architecture

```
                       Runnable
                            │
   ┌──────────┬─────────────┼───────────────┬────────────────────┐
   ▼          ▼             ▼               ▼                    ▼
 Prompt      Model      RunnableLambda  RunnablePassthrough  RunnableParallel
                                            └── .assign()    (dict of Runnables)
```

`RunnablePassthrough` almost always appears **inside a parallel dict**: some keys compute new things,
`RunnablePassthrough()` keeps the input. `.assign()` is sugar that merges "keep everything" with "add new
keys" in one step.

---

## 5. Internal Working — the three behaviors side by side

```
  input = {"question": "What is LCEL?"}

  ── RunnablePassthrough() ───────────────────────────────────────────
     returns the input UNCHANGED
     → {"question": "What is LCEL?"}

  ── {"context": retriever, "question": RunnablePassthrough()}  (parallel) ──
     each branch runs on the SAME input; results collected by key
     → {"context": [...docs...], "question": "What is LCEL?"}
        (retriever ran on the input; passthrough echoed it back)

  ── RunnablePassthrough.assign(context=retriever) ───────────────────
     KEEP the whole input, ADD "context"
     → {"question": "What is LCEL?", "context": [...docs...]}
```

The distinction that trips people up:

```
  {"context": retriever}                         → {"context": ...}                    (ONLY new key)
  RunnablePassthrough.assign(context=retriever)  → {**input, "context": ...}           (keep + add)
```

Use a **plain dict** when you want a brand-new shape; use **`.assign()`** when you must **retain the
existing fields** too.

---

## 6. The three forms

### RunnablePassthrough() — echo input unchanged

**Definition:** Returns exactly what it receives.

**Why it exists:** Keep the original input as one branch of a parallel step.

**When developers use it:** The RAG fan-out; anywhere a later step needs the raw input.

**Real-life use case:** The plain conveyor segment carrying the item forward.

```python
from langchain_core.runnables import RunnablePassthrough

rag_inputs = {
    "context":  retriever,
    "question": RunnablePassthrough(),
}
# rag_inputs.invoke("What is LCEL?") → {"context": [...], "question": "What is LCEL?"}
```

---

### RunnablePassthrough.assign(**kwargs) — keep input AND add fields

**Definition:** Merges the input dict with new keys computed by the given Runnables.

**Why it exists:** Progressively enrich the data while keeping everything already there.

**When developers use it:** Multi-stage chains, RAG, classify-then-route (add a `topic` field, keep the
original input).

**Real-life use case:** Stamping an extra box on the form without erasing the rest.

```python
from langchain_core.runnables import RunnablePassthrough

enriched = RunnablePassthrough.assign(summary=summarize_chain)
enriched.invoke({"text": article})
# → {"text": article, "summary": "..."}   ← both available downstream
```

> ⚠️ `.assign()` requires the input to be a **dict** (it merges keys into it). Each value is a Runnable
> (or auto-wrapped function) that receives the whole input dict.

---

### RunnablePassthrough(func) / with side effects

**Definition:** You can pass a function to run for **side effects** (e.g., logging) while still returning
the input unchanged.

**Why it exists:** Peek at data mid-chain without altering it.

**When developers use it:** Debugging/logging inside a pipeline.

```python
RunnablePassthrough(lambda x: print("passing:", x)).invoke({"q": "hi"})
# prints, then returns {"q": "hi"} unchanged
```

---

## 7. The canonical RAG chain (why this matters)

This is the pattern you'll write again and again — it relies entirely on passthrough/assign:

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY this context.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}   # keep question, fetch context
    | prompt                                                    # prompt reads {context, question}
    | model
    | StrOutputParser()
)

rag_chain.invoke("What is LCEL?")
```

Without `RunnablePassthrough()`, the question would be replaced by the retriever's output and the prompt
couldn't fill `{question}`.

---

## 8. assign + itemgetter (feed exact fields onward)

`.assign()` grows the dict; `operator.itemgetter` then selects just what the next step needs:

```python
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(summary=summarize_chain)   # {"text","language","summary"}
    | {                                                   # pick fields for the translate prompt
        "summary":  itemgetter("summary"),
        "language": itemgetter("language"),
      }
    | translate_chain
)
```

---

## 9. Quick reference (how the two files fit together)

| Tool | Input → Output | Use it to |
|------|----------------|-----------|
| `RunnableLambda(f)` (file 1) | `x → f(x)` | Run custom logic / reshape data |
| `RunnablePassthrough()` | `x → x` | Keep the original input (esp. in a parallel dict) |
| `RunnablePassthrough.assign(k=r)` | `x → {**x, k: r(x)}` | Keep everything **and** add a field |
| `{ "k": r, ... }` (parallel dict) | `x → {"k": r(x), ...}` | Build a **new** dict shape (replaces input) |
| `itemgetter("k")` | `x → x["k"]` | Select one field to pass on |

In [1]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

chain = RunnableParallel(name=RunnablePassthrough(), message=lambda x: f"Hello, {x['name']}!")

result = chain.invoke({"name": "Sachin"})

print(result)

{'name': {'name': 'Sachin'}, 'message': 'Hello, Sachin!'}


In [2]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

chain = RunnableParallel(user=RunnablePassthrough(),
                         greeting=RunnableLambda(lambda x: f"Hello {x['name']}!"),
                         info=RunnableLambda(lambda x: f"{x['name']} is {x['age']} years old and lives in {x['city']}.")
                        )

result = chain.invoke({"name": "Sachin", "age": 25, "city": "Delhi"})

print(result)

{'user': {'name': 'Sachin', 'age': 25, 'city': 'Delhi'}, 'greeting': 'Hello Sachin!', 'info': 'Sachin is 25 years old and lives in Delhi.'}


In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough

joke_prompt = PromptTemplate(template='Write a joke about {topic}',
                         input_variables=['topic']
)

llm = ChatOllama(model="qwen3:8b")

parser = StrOutputParser()

explain_joke_prompt = PromptTemplate(template='Explain the following joke - {text}',
                         input_variables=['text']
)

joke_gen_chain = RunnableSequence(joke_prompt, llm, parser)

parallel_chain = RunnableParallel({'joke': RunnablePassthrough(),
                                   'explanation': RunnableSequence(explain_joke_prompt, llm, parser)})

final_chain = joke_gen_chain | parallel_chain

print(final_chain.invoke({'topic':'cricket'}))

{'joke': 'Why did the cricket go to the doctor?  \nBecause it was feeling a bit **stump-ed**! 🏏😄  \n\n*(Bonus: The doctor prescribed a few "runs" to help it recover!)*', 'explanation': 'The joke plays on the dual meanings of the word **"stump"** and the sport of **cricket**:\n\n1. **"Stump-ed"** is a pun on **"stumped"** (meaning confused or stuck) and **"stump"** (a part of the cricket equipment, the three vertical posts in the wicket). The cricket (the insect) went to the doctor because it was feeling stuck or confused ("stump-ed"), blending the insect\'s natural context with the sport\'s terminology.\n\n2. The **bonus line** adds another layer: the doctor prescribes **"runs"**, which in cricket (the sport) refers to the points scored by the batsman. This ties back to the sport\'s jargon, humorously suggesting the cricket needs to "score runs" to recover, reinforcing the pun on the sport\'s terms.\n\n**Why it\'s funny**: The joke cleverly merges the insect (cricket) and the sport (cr